<h1 style="text-align: center; margin-bottom: 0;">Deep Learning based Recommendation System</h1>
<h2 style="text-align: center; margin-top: 35px;">Sravya Kodati, Rutvik Dhopate, Vivek Rachakonda</h2>

<center  style="font-size:22px; margin-top: 50px;">NRCMA - Neural Recommendation with Cross-Modality Mutual Attention (Luo, et al.)</center>


<ul style="font-size:20px;">
    <li>Recap of problem statement</li>
    <li>Architecture</li>
    <li>Libraries</li>
    <li>Preprocessing</li>
    <li>PyTorch Model Class</li>
    <li>Training Loop</li>
    <li>Next Steps</li>
</ul>

## Architecture

<details>
    <div style="text-align: center;">
    <img src="../assets/images/nrcma_architecture.png"></img>   
    </div>
</details>

## Libraries

In [82]:
import math
import yaml
import torch
import random

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
from torchinfo import summary
import matplotlib.pyplot as plt
from dataclasses import dataclass

import gensim.downloader as api
from sklearn.model_selection import train_test_split

In [80]:
df = pd.read_csv("../data/books_mini_sample.csv")

In [67]:
filtered_df = df[['rating', 'text', 'user_id', 'asin']].copy()
filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,AE224GVO7OHTYF26U6ER6BEVIUAQ,0486241386
1,5.0,Ladies 'n Gentlemen...we have another hit from...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000MGATTE
2,5.0,I ordered this book for my Kindle. Koontz and ...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UDNBRQ
3,5.0,Absolutely loved this book! Stephen King just...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UZJREU
4,5.0,Another King satisfied reader~! what can I sa...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B001RF3U9K


In [68]:
filtered_df['user_id'] = filtered_df['user_id'].astype('category').cat.codes
filtered_df['asin'] = filtered_df['asin'].astype('category').cat.codes

filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,0,1655
1,5.0,Ladies 'n Gentlemen...we have another hit from...,1,5766
2,5.0,I ordered this book for my Kindle. Koontz and ...,1,5817
3,5.0,Absolutely loved this book! Stephen King just...,1,5819
4,5.0,Another King satisfied reader~! what can I sa...,1,5877


### Load Glove vectors

In [10]:
word2vec_model = api.load('word2vec-google-news-300')

# Access word embeddings from the pre-trained model
vector = word2vec_model['word']

In [11]:
# adding a padding token
word2vec_model['<pad>'] = np.zeros(300)

## Preprocessing

In [69]:
%%time
max_length = 100
filtered_df['text'] = filtered_df['text'].astype('string')\
                                        .fillna('')\
                                        .str.split(' ')\
                                        .apply(lambda words: [word for word in words if word in word2vec_model.key_to_index])

filtered_df['text'] = filtered_df['text'].apply(
    lambda x: x[:max_length] + ['<pad>'] * (max_length - len(x)) if len(x) < max_length else x[:max_length]
)

CPU times: user 1.44 s, sys: 436 ms, total: 1.87 s
Wall time: 2.11 s


In [70]:
filtered_df['rating'] = (filtered_df['rating'] - filtered_df['rating'].min()) / (filtered_df['rating'].max() - filtered_df['rating'].min())

In [71]:
filtered_df[['text', 'rating']].head()

,text,rating
0,"[This, is, great, book, with, lot, ONE, CAUTIO...",1.0
1,"[Ladies, have, another, hit, from, Stephen, th...",1.0
2,"[I, ordered, this, book, for, my, Koontz, two,...",1.0
3,"[Absolutely, loved, this, Stephen, King, just,...",1.0
4,"[Another, King, satisfied, what, can, I, The, ...",1.0


## Loading config file

In [72]:
# Load Configuration

with open('../config/nrcma.yaml') as f:
    config = yaml.safe_load(f)

### Fix all the seeds for reproducibility

In [73]:
def seed_everything(seed=42):
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    # this may hurt the performance
    torch.backends.cudnn.benchmark = False


# Fix all the seeds for reproducibility
seed_everything(seed=config['training']['random_seed'])

### WandB for logging and checkpointing

In [87]:
import wandb

wandb_project = 'deep_rec_sys_user_reviews'

In [98]:
run = wandb.init(project=wandb_project, 
                 config=config, 
                 tags=["run_without_batches", "overfitting_single_set"],
                 name="nrcma_run_2",
                 notes="This is the second run of NRCMA! Trying to see if we can overfit a single set of 1000 records!!")

## PyTorch Dataset Class

In [94]:
%%time
from torch.utils.data import Dataset, DataLoader


class RecSysDataset(Dataset):
    def __init__(self, df, word2vec_model, max_reviews):
        self.df = df
        self.word2vec_model = word2vec_model
        self.max_reviews = max_reviews

    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        user_reviews = self.df[self.df['user_id'] == row.user_id]['text']
        w2v_user_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in user_reviews if len(i)!=0]
        user_tower_input = torch.stack(w2v_user_tensor, dim=0)

        item_reviews = self.df[self.df['asin'] == row.asin]['text']
        w2v_item_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in item_reviews if len(i)!=0]
        item_tower_input = torch.stack(w2v_item_tensor, dim=0)

        target_size = (self.max_reviews, max_length, 300)
        user_tower_input = self._resize_tensors(user_tower_input, target_size).unsqueeze(1)
        item_tower_input = self._resize_tensors(item_tower_input, target_size).unsqueeze(1)
        
        user_id = torch.tensor([row['user_id']], dtype=torch.int)
        asin = torch.tensor([row['asin']], dtype=torch.int)
        rating = torch.tensor([row['rating']], dtype=torch.float)

        return user_tower_input, item_tower_input, rating, user_id, asin

    def _resize_tensors(self, input_tensor, target_size):

        # Resize the num_reviews if needed; TODO - change to random sampling? picking only first `max_reviews` number of reviews of the user or item
        tensor_size = input_tensor.shape
        if tensor_size[0] != target_size[0]:
            if tensor_size[0] < target_size[0]:
                diff = target_size[0] - tensor_size[0]
                padding = torch.zeros((diff, tensor_size[1], tensor_size[2]))
                output_tensor = torch.cat((input_tensor, padding), dim=0)
            else:
                output_tensor = input_tensor[:target_size[0]]
        else:
            output_tensor = input_tensor
        return output_tensor



CPU times: user 78 μs, sys: 48 μs, total: 126 μs
Wall time: 129 μs


In [83]:
filtered_df.columns

Index(['rating', 'text', 'user_id', 'asin'], dtype='object')

In [99]:
%%time

# lets split the data into train, test, val - its 2025 and sklearn still doesn't support val split in its function!! :(
X_train, X_test = train_test_split(filtered_df, test_size=0.05, random_state=42)
X_train, X_val = train_test_split(X_train, test_size=(0.05/0.95), random_state=42)

train_recsys_dataset = RecSysDataset(X_train, word2vec_model, 10)
val_recsys_dataset = RecSysDataset(X_val, word2vec_model, 10)
test_recsys_dataset = RecSysDataset(X_test, word2vec_model, 10)

train_dataloader = DataLoader(train_recsys_dataset, batch_size=32, shuffle=True)

for i, sample in enumerate(train_recsys_dataset):
    if i > 2:
        break
    user_tower_input, item_tower_input, label, user_id, asin = sample
    print(user_tower_input.shape, item_tower_input.shape, label.shape, user_id.shape, asin.shape)
    

torch.Size([10, 1, 100, 300]) torch.Size([10, 1, 100, 300]) torch.Size([1]) torch.Size([1]) torch.Size([1])
torch.Size([10, 1, 100, 300]) torch.Size([10, 1, 100, 300]) torch.Size([1]) torch.Size([1]) torch.Size([1])
torch.Size([10, 1, 100, 300]) torch.Size([10, 1, 100, 300]) torch.Size([1]) torch.Size([1]) torch.Size([1])
CPU times: user 95.6 ms, sys: 63.8 ms, total: 159 ms
Wall time: 93 ms


## NRCMA Module

In [100]:
# creating a dataclass for the model

@dataclass
class NRCMAConfig:
    num_users: int
    num_products: int
    num_conv_filters: int
    word_embed_dim: int
    id_embed_dim: int
    attention_vector_dim: int
    feature_vector_dim: int

    @staticmethod
    def from_config(config):
        return NRCMAConfig(**config)


In [101]:
nrcma_config = NRCMAConfig.from_config(config['model'])

In [102]:
class NRCMA(nn.Module):
    def __init__(self, config: NRCMAConfig):
        super(NRCMA, self).__init__()

        # Extract all config parameters into local variables
        num_users = config.num_users
        num_products = config.num_products
        id_embed_dim = config.id_embed_dim
        num_conv_filters = config.num_conv_filters
        word_embed_dim = config.word_embed_dim
        attention_vector_dim = config.attention_vector_dim
        feature_vector_dim = config.feature_vector_dim
        
        self.user_embedding = nn.Embedding(num_users, id_embed_dim)
        self.item_embedding = nn.Embedding(num_products, id_embed_dim)

        # user tower
        self.user_cnn = nn.Conv2d(in_channels=1, out_channels=num_conv_filters, kernel_size=(1, word_embed_dim), padding=0, bias=True)
        self.word_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.word_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)
        self.review_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.review_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)

        # item tower
        self.item_cnn = nn.Conv2d(in_channels=1, out_channels=num_conv_filters, kernel_size=(1, word_embed_dim), padding=0, bias=True)
        self.item_word_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.item_word_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)
        self.item_review_level_matrix = nn.Linear(id_embed_dim, attention_vector_dim)
        self.item_review_harmony_matrix = nn.Linear(num_conv_filters, attention_vector_dim)

        # factorization machine, 2 * num_conv_filters as we get concat features from user, item
        self.fm_linear = nn.Linear(2*num_conv_filters, 1)
        self.v = nn.Parameter(torch.empty(2*num_conv_filters, feature_vector_dim))
        
        # Initialize self.v similar to nn.Linear weights
        nn.init.kaiming_uniform_(self.v, a=math.sqrt(5))


    def forward(self, user_input, item_input, user_id, item_id):
        
        user_embedding = self.user_embedding(user_id)
        item_embedding = self.item_embedding(item_id)

        d_u = self.process_single_tower(user_input, item_embedding, tower='user')
        d_i = self.process_single_tower(item_input, user_embedding, tower='item')
        o = torch.cat((d_u, d_i), dim=1)

        # factorization machine
        prediction = self.fm_linear(o) + (o @ torch.triu(self.v @ self.v.T, diagonal=1) @ o.T)
        return prediction

    def process_single_tower(self, input_matrix, embedding, tower):

        if tower == 'user':
            cnn = self.user_cnn
            word_level_matrix = self.word_level_matrix
            word_harmony_matrix = self.word_harmony_matrix
            review_level_matrix = self.review_level_matrix
            review_harmony_matrix = self.review_harmony_matrix
        else:
            cnn = self.item_cnn
            word_level_matrix = self.item_word_level_matrix
            word_harmony_matrix = self.item_word_harmony_matrix
            review_level_matrix = self.item_review_level_matrix
            review_harmony_matrix = self.item_review_harmony_matrix

        # process information through single tower - employing cross attention
        review_features = F.relu(cnn(input_matrix))
        beta_k = F.relu(word_level_matrix(embedding))
        
        intermediate_output_1 = word_harmony_matrix(review_features.squeeze().transpose(-1, -2)).transpose(-1, -2) # not sure if this step is correct; need to check this
        b_c = beta_k @ intermediate_output_1
        alpha_c = nn.Softmax(dim=2)(b_c)
        
        d_uk = review_features.squeeze() @ torch.transpose(alpha_c, -1, -2)

        beta_u = F.relu(review_level_matrix(embedding))
        intermediate_output_2 = review_harmony_matrix(d_uk.squeeze())
        b_k = beta_u @ intermediate_output_2.T

        alpha_k = nn.Softmax(dim=1)(b_k)
        d_u = alpha_k @ d_uk.squeeze()

        return d_u


In [103]:
model = NRCMA(nrcma_config)

## Model Summary

<span style="font-size:20px;">92.5% of parameters are present in the embedding layers</span>

In [41]:
summary(model)

Layer (type:depth-idx)                   Param #
NRCMA                                    1,600
├─Embedding: 1-1                         491,940
├─Embedding: 1-2                         236,670
├─Conv2d: 1-3                            24,080
├─Linear: 1-4                            620
├─Linear: 1-5                            1,620
├─Linear: 1-6                            620
├─Linear: 1-7                            1,620
├─Conv2d: 1-8                            24,080
├─Linear: 1-9                            620
├─Linear: 1-10                           1,620
├─Linear: 1-11                           620
├─Linear: 1-12                           1,620
├─Linear: 1-13                           161
Total params: 787,491
Trainable params: 787,491
Non-trainable params: 0

In [104]:
config['training']

{'batch_size': 64,
 'learning_rate': 0.001,
 'num_epochs': 10,
 'random_seed': 42,
 'log_interval': 10,
 'checkpoint_dir': 'checkpoints/'}

In [105]:
args = config['training']

optimizer = optim.Adam(model.parameters(), lr=args['learning_rate'])
model = NRCMA(nrcma_config)

loss_fn = nn.MSELoss()

## PyTorch Training Loop

In [106]:
%%time
total_loss = []
max_recs = 1_000
args['num_epochs'] = 20 # TODO - remove this line; only for testing

for epoch in range(args['num_epochs']):

    model.train()
    for i, sample in enumerate(train_recsys_dataset):
        if i > max_recs:
            break
        optimizer.zero_grad(set_to_none=True)
        user_tower_input, item_tower_input, true_rating, user_id, asin = sample
        predicted_rating = model(user_tower_input, item_tower_input, user_id, asin)
        train_loss = loss_fn(predicted_rating, true_rating)

        train_loss.backward()
        optimizer.step()
        total_loss.append(train_loss.item())

        if i < max_recs:
            run.log({"train/train_loss":train_loss.item(), "train/epoch": epoch})

        # run.log({"train/train_loss":train_loss.item(), "train/epoch": epoch})

    model.eval()
    val_loss = 0
    for j, sample in enumerate(val_recsys_dataset):
        user_tower_input, item_tower_input, true_rating, user_id, asin = sample
        prediction = model(user_tower_input, item_tower_input, user_id, asin)
        loss = loss_fn(prediction, true_rating)
        val_loss += loss.item()

    if i == max_recs+1:
        run.log({"train/train_loss":train_loss.item(), "train/epoch":epoch, "val/val_loss":val_loss})

    print(f'avg val loss - {val_loss/len(val_recsys_dataset)}')


/Users/vivekrachakonda/Documents/Courses/Capstone/github/deep-rec-sys-amazon-reviews/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


CPU times: user 14min 44s, sys: 1min 34s, total: 16min 19s
Wall time: 4min 10s


In [107]:
torch.save({
        # 'epoch': current_epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, 'nrcma_first_run.pt')
run.log_model("./nrcma_first_run.pt", "nrcma_first_run")
run.finish()

train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/train_loss,█▅▂█████▅█▅██████▅▂█▁███████▅▅▅▂█▅█▅█▂▅█
train/epoch,19
train/train_loss,1.01403


## Next Steps

<ul style="font-size:18px;">
    <li>Split this notebook into 3 scripts - utils (data loading), model class, training</li>
    <li>Retraining - starting from a intermediate checkpoint</li>
    <li>Perform Hyperparameter Tuning </br>
        - conv filters </br>
        - attention vector dimension
    </li>
    <li>Train the model on GPU</li>
    <li>Scaling the training to multiple GPUs</li>
</ul>